In [9]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

load_dotenv()


True

In [10]:
llm=ChatGroq(model="qwen/qwen3-32b", temperature=0)

@tool
def get_customer_info_tool(customer_name: str) -> str:
    """ 
        Fetches customer information based on a given customer name.
        This is a placeholder function that simulates retrieving customer data.
        In a real implementaion, this would query a database or external API
    """

    #simulated customer data.

    customers={
        "Krishna":{
            "email":"krishna_001@gmail.com",
            "credit_card": "4111-1111-1111-1111",
            "name":"Krishna",
            "loyalty_status":"Gold"            
        },
        "Alice":{
            "email":"alice_002@gmail.com",
            "credit_card": "5500-0000-0000-0004",
            "name":"Alice",
            "loyalty_status":"Silver"            
        },
        "Bob":{
            "email":"bob2009@gmail.com",
            "credit_card": "6011-1111-1111-1117",
            "name":"Bob",
            "loyalty_status":"Bronze"            
        },
        
    }

    return customers.get(customer_name,"Customer not found")    
    

In [16]:
SYSTEM_PROMPT="""You are a customer service assistant.
        you have access to get_customer_info_tool which provides customer information based on the customer name.
        when a user asks information about a customer, user the get_customer_info_tool to retrieve.

        IMPORTANT RULES:
        1. Return all data fields EXACTLY as received from the tool- do not reformat.
        2. Credit card details must always be returned in their original format with dashes
            e.g., XXXX-XXXX-XXXX-XXXX  - never remove dashes or spaces.
        3. do not handle PII directly. Middleware will automatically redact/mask sensitive fields- your job is to pass the raw values through unchanged.
        4. Return information as plain text, Not JSON.
    """
agent = create_agent(
    model = llm,
    tools=[get_customer_info_tool],
    system_prompt=SYSTEM_PROMPT,
    middleware = [
        #Mask credi cards
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_tool_results = True
        ),
        #Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_tool_results = True,
            apply_to_output = True
            
        )
    ]
)

In [21]:
#when user provides PII, it will be handles according to the strategy.

result = agent.invoke({
    "messages":[{
        "role": "user",
        "content": "Give me information about customer Bob"
    }]
})

print(result["messages"][-1].content)

Email: [REDACTED_EMAIL]  
Credit Card: ****-****-****-1117  
Name: Bob  
Loyalty Status: Bronze
